# 38 — Deal Room Analyst
## Confidence-Gated Sequential M&A Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/38-deal-room-analyst/deal_room_analyst_workbook.ipynb)

---

### What this example teaches

How to build a **confidence-gated sequential pipeline** — one where each stage scores its own certainty and the orchestrator halts rather than forwarding weak signals downstream. This is the pattern that prevents a half-baked contract review from polluting a financial model or a board memo.

### The business problem

A private equity deal desk receives dozens of packages a month: a term sheet, a handful of management accounts, a data room, and a financial summary. Each package needs:

1. A contract risk review (red-line candidates, missing protections)
2. A due diligence report (risk register across financial, legal, operational, management, regulatory areas)
3. A financial model (3-year P&L projection + implied valuation)
4. A board memo (go / pause / reject with conditions)

Running all four stages blindly on thin data wastes analyst time and produces unreliable board-facing outputs. The orchestrator in this example solves that by requiring each stage to self-report a `confidence` score (0.0-1.0). If a stage scores below **0.6**, the pipeline halts, raises an `EscalationFlag`, and returns a partial result — it does not hand weak signals to expensive downstream stages.

### Harness focus

**Confidence-gated sequential pipeline** — structured outputs carry a numeric `confidence` field; the orchestrator inspects it after each stage call and either advances or halts.

### Prerequisites

- [#9 Contract Reviewer](../9-contract-reviewer/) — clause-grounded risk findings
- [#10 Due Diligence](../10-due-diligence/) — unified risk register pattern
- [#13 M&A Screener](../13-ma-screener/) — deal qualification
- [#19 Financial Modeller](../19-financial-modeller/) — projection schemas
- [#35 Board Memo Synthesizer](../35-board-memo-synthesizer/) — multi-source synthesis

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | Framework comparison — why raw openai-sdk here |
| 2 | Schema — typed inputs and outputs for every stage |
| 3 | Prompts — specialist system prompts per stage |
| 4 | Workflow — the confidence gate pattern |
| 5 | Clean deal — NovaTech (pipeline completes) |
| 6 | Messy deal — Acme Logistics (pipeline halts) |
| * | Starter exercise + answer key |

---

## Part 1 — Framework Comparison

The same confidence-gate pattern appears across every major agent framework. Here is how it maps:

| Framework | Mechanism | How the gate works |
|-----------|-----------|--------------------|
| **Raw openai-sdk** (this demo) | `response_format` with `json_schema` | Orchestrator function inspects `.confidence` after each call; returns early if below threshold |
| **LangGraph** | Conditional edges | A routing function reads the confidence field on the state object and routes to `HALT` or the next node |
| **LangChain** | Sequential chains + output parsers | `RunnableBranch` or a custom `Runnable` checks the parsed output before passing to the next chain |
| **CrewAI** | Task dependencies with guardrails | A task's `guardrail` callback returns `(False, reason)` to block downstream tasks when confidence is low |

### Why raw openai-sdk in this demo?

Three reasons:

1. **Clarity** — the gate logic is a single `if` statement you can read in one glance. No framework abstractions hide what is happening.
2. **Portability** — the workflow function is framework-agnostic Python; you can wrap it in any orchestrator later.
3. **`response_format` structured output** — OpenAI's JSON schema enforcement guarantees the `confidence` field is always present and numeric, which the gate depends on.

When you outgrow this pattern — parallel stage execution, human-in-the-loop approvals, persistent state across sessions — reach for LangGraph's conditional edges or CrewAI's guardrail callbacks.

In [ ]:
!pip install openai pydantic python-dotenv

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
    print("API key loaded from .env.")

---

## Part 2 — Schema

Every stage in the pipeline has its own typed output model. The schema design has two key properties that enable the confidence gate:

1. **`confidence: float`** — present on `ContractReview`, `DueDiligenceReport`, `FinancialModel`, and `BoardMemo`. The orchestrator reads this field after each call. The system prompts define exactly what 0.0, 0.5, and 1.0 mean for each stage — so the model has a calibration reference, not a vague instruction.

2. **`EscalationFlag`** — a structured record of which stage halted, the confidence that triggered it, the threshold, and a plain-language reason. This is what a human reviewer sees instead of an error.

3. **`DealRoomResult`** — the single return type for all outcomes. `completed=True` means the board memo is ready; `completed=False` means `escalation` is populated and downstream fields are `None`.

This means callers always get the same envelope regardless of whether the pipeline ran to completion or halted mid-way.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class DealInput(BaseModel):
    company_name: str = Field(description="Target company name")
    contract_text: str = Field(description="Full text of the commercial agreement or term sheet")
    diligence_documents: list[str] = Field(
        description="List of due diligence document excerpts (financials, management bio, etc.)"
    )
    financial_summary: str = Field(
        description="Plain-text summary of financials: revenue, EBITDA, debt, key assumptions"
    )


class RiskFinding(BaseModel):
    clause: str = Field(description="Clause or section the risk was found in")
    severity: Literal["low", "medium", "high", "critical"] = Field(description="Risk severity")
    description: str = Field(description="Plain-language description of the risk")
    mitigation: str = Field(description="Suggested mitigation or negotiation point")


class ContractReview(BaseModel):
    company_name: str = Field(description="Target company name")
    risk_findings: list[RiskFinding] = Field(description="Ranked list of risk findings")
    missing_protections: list[str] = Field(description="Standard protections absent from the contract")
    recommended_redlines: list[str] = Field(description="Top redline changes to negotiate")
    confidence: float = Field(description="Analyst confidence score 0.0-1.0 based on completeness of contract text")
    summary: str = Field(description="Two-sentence plain-language summary of overall contract risk")


class DiligenceRisk(BaseModel):
    area: str = Field(description="Risk area: financial, legal, operational, management, regulatory")
    severity: Literal["low", "medium", "high", "critical"] = Field(description="Risk severity")
    likelihood: Literal["low", "medium", "high"] = Field(description="Probability of materialising")
    description: str = Field(description="Plain-language description of the risk")
    mitigation: str = Field(description="Recommended mitigation action")


class DueDiligenceReport(BaseModel):
    company_name: str = Field(description="Target company name")
    risks: list[DiligenceRisk] = Field(description="Unified risk register ranked by severity")
    deal_breakers: list[str] = Field(description="Risks severe enough to halt the deal")
    confidence: float = Field(description="Analyst confidence score 0.0-1.0 based on document completeness")
    summary: str = Field(description="Two-sentence summary of overall due diligence findings")


class FinancialModel(BaseModel):
    company_name: str = Field(description="Target company name")
    revenue_year1: float = Field(description="Year 1 projected revenue in USD")
    revenue_year2: float = Field(description="Year 2 projected revenue in USD")
    revenue_year3: float = Field(description="Year 3 projected revenue in USD")
    ebitda_year3: float = Field(description="Year 3 projected EBITDA in USD")
    implied_valuation: float = Field(description="Implied enterprise value at entry multiple in USD")
    key_assumptions: list[str] = Field(description="Top 3-5 assumptions driving the model")
    confidence: float = Field(description="Model confidence score 0.0-1.0 based on data quality")
    commentary: str = Field(description="Two-sentence commentary on the financial outlook")


class BoardMemo(BaseModel):
    company_name: str = Field(description="Target company name")
    recommended_position: Literal["proceed", "pause", "reject"] = Field(
        description="Board recommendation based on all stage findings"
    )
    executive_summary: str = Field(description="2-3 paragraph summary for board directors")
    key_risks: list[str] = Field(description="Top 3-5 risks distilled from all stages")
    conditions_to_proceed: list[str] = Field(
        description="Conditions that must be met before proceeding; empty if rejecting"
    )
    one_sentence_verdict: str = Field(description="Single sentence board-ready verdict")
    confidence: float = Field(description="Overall pipeline confidence score 0.0-1.0")


class EscalationFlag(BaseModel):
    stage: str = Field(description="Stage that triggered the halt: contract_review, due_diligence, financial_model")
    confidence: float = Field(description="Confidence score that fell below threshold")
    threshold: float = Field(description="Threshold that was not met")
    reason: str = Field(description="Plain-language reason the stage was flagged for escalation")


class DealRoomResult(BaseModel):
    company_name: str = Field(description="Target company name")
    completed: bool = Field(description="True if all stages passed confidence gates; False if halted")
    escalation: EscalationFlag | None = Field(
        default=None, description="Set when the pipeline halted before reaching the board memo"
    )
    contract_review: ContractReview | None = Field(default=None)
    due_diligence: DueDiligenceReport | None = Field(default=None)
    financial_model: FinancialModel | None = Field(default=None)
    board_memo: BoardMemo | None = Field(default=None)


print("Schema defined.")
print("Stages with confidence gate: ContractReview, DueDiligenceReport, FinancialModel")
print(f"Pipeline result fields: {list(DealRoomResult.model_fields.keys())}")

---

## Part 3 — Prompts

Each stage has a dedicated specialist system prompt. The critical design choice is the **confidence calibration instruction** embedded in every prompt.

Without calibration, `confidence` is arbitrary. With it, the model has a reference scale:

| Stage | 1.0 (high confidence) | 0.5 (medium) | 0.3 or below (low) |
|-------|-----------------------|--------------|--------------------|
| Contract Review | Full signed agreement | Term sheet | Fragmentary text |
| Due Diligence | Full data room | Management accounts only | Press releases / public data |
| Financial Model | Audited financials | Management accounts | Verbal summary only |

The board memo prompt instructs the model to average the three stage confidence scores — so it inherits and compounds the earlier assessments.

In [ ]:
CONTRACT_SYSTEM = (
    "You are a senior M&A contracts lawyer. Review the commercial agreement provided and produce a "
    "ContractReview JSON object.\n\n"
    "Rules:\n"
    "- Identify every material risk clause and classify severity: critical (deal-breaker), high (requires redline), "
    "medium (negotiate if possible), low (acceptable with monitoring).\n"
    "- List standard protections that are absent (indemnities, IP assignment, limitation of liability, etc.).\n"
    "- Rank recommended_redlines from most to least important.\n"
    "- Set confidence based on document completeness: 1.0 = full signed agreement, 0.5 = term sheet, "
    "0.3 or below = fragmentary.\n"
    "- Return ONLY valid JSON matching the schema -- no prose."
)

DILIGENCE_SYSTEM = (
    "You are a lead due diligence analyst at a private equity firm. Review the provided documents and produce a "
    "DueDiligenceReport JSON object.\n\n"
    "Rules:\n"
    "- Cover financial, legal, operational, management, and regulatory risk areas.\n"
    "- Rate each risk by severity (critical/high/medium/low) and likelihood (high/medium/low).\n"
    "- List deal_breakers only for risks that would cause a rational acquirer to walk away.\n"
    "- Set confidence based on document completeness: 1.0 = full data room, 0.6 = management accounts only, "
    "0.3 = press releases and public data.\n"
    "- Return ONLY valid JSON matching the schema -- no prose."
)

FINANCIAL_SYSTEM = (
    "You are a corporate finance analyst. Read the financial summary provided and produce a FinancialModel "
    "JSON object with a 3-year revenue and EBITDA projection.\n\n"
    "Rules:\n"
    "- Revenue projections must be consistent with the stated business model and growth rates.\n"
    "- EBITDA year 3 must reflect realistic margin expansion from base.\n"
    "- implied_valuation = EBITDA year 3 x entry multiple (use 8x if not stated).\n"
    "- List the top 3-5 assumptions that drive the model.\n"
    "- Set confidence: 1.0 = audited financials, 0.7 = management accounts, 0.4 = verbal summary only.\n"
    "- Return ONLY valid JSON matching the schema -- no prose."
)

BOARD_SYSTEM = (
    "You are a chief strategy officer preparing a board memo for an M&A decision. You have received a contract "
    "review, due diligence report, and financial model. Synthesise them into a BoardMemo JSON object.\n\n"
    "Rules:\n"
    "- recommended_position: 'proceed' if all stages are clean and model is credible; 'pause' if material risks "
    "exist but are manageable; 'reject' if deal-breakers are present or confidence is too low.\n"
    "- executive_summary: 2-3 paragraphs citing findings from all three stages.\n"
    "- key_risks: top 3-5 risks drawn from contract and diligence stages.\n"
    "- conditions_to_proceed: specific conditions that must be satisfied before closing.\n"
    "- confidence = average of the three stage confidence scores.\n"
    "- Return ONLY valid JSON matching the schema -- no prose."
)

print("Prompts defined.")
print("Confidence calibration anchors embedded in: CONTRACT_SYSTEM, DILIGENCE_SYSTEM, FINANCIAL_SYSTEM")

---

## Part 4 — Workflow: The Confidence Gate Pattern

The core pattern is a **short-circuit sequential pipeline**. Each stage:

1. Calls the LLM with a specialist system prompt and structured output schema
2. Parses the typed response
3. Checks `result.confidence < threshold`
4. If below threshold: returns a `DealRoomResult(completed=False, escalation=EscalationFlag(...))`
5. If above threshold: continues to the next stage

The gate runs three times (contract -> diligence -> financials). The board memo does not have its own gate because it synthesises from already-gated inputs.

```
DealInput
    |
    v
[contract_review]  confidence < 0.6? --> EscalationFlag (halt)
    |
    v
[due_diligence]    confidence < 0.6? --> EscalationFlag (halt)
    |
    v
[financial_model]  confidence < 0.6? --> EscalationFlag (halt)
    |
    v
[board_memo]  --> DealRoomResult(completed=True)
```

The `_call` helper uses OpenAI's `response_format` with `json_schema` and `strict=True`. This guarantees the JSON matches the schema exactly — no missing `confidence` field, no type coercion surprises.

In [ ]:
import json
import os

from openai import OpenAI

_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
_MODEL = "gpt-4o-mini"
_CONFIDENCE_THRESHOLD = 0.6


def _call(system: str, user: str, schema_name: str, schema: dict) -> str:
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": schema_name, "strict": True, "schema": schema},
        },
    )
    return resp.choices[0].message.content


def _review_contract(deal: DealInput) -> ContractReview:
    raw = _call(
        CONTRACT_SYSTEM,
        deal.contract_text,
        "ContractReview",
        ContractReview.model_json_schema(),
    )
    return ContractReview.model_validate_json(raw)


def _run_due_diligence(deal: DealInput) -> DueDiligenceReport:
    raw = _call(
        DILIGENCE_SYSTEM,
        json.dumps({"company": deal.company_name, "documents": deal.diligence_documents}),
        "DueDiligenceReport",
        DueDiligenceReport.model_json_schema(),
    )
    return DueDiligenceReport.model_validate_json(raw)


def _build_financial_model(deal: DealInput) -> FinancialModel:
    raw = _call(
        FINANCIAL_SYSTEM,
        json.dumps({"company": deal.company_name, "financial_summary": deal.financial_summary}),
        "FinancialModel",
        FinancialModel.model_json_schema(),
    )
    return FinancialModel.model_validate_json(raw)


def _write_board_memo(
    deal: DealInput,
    contract: ContractReview,
    diligence: DueDiligenceReport,
    financials: FinancialModel,
) -> BoardMemo:
    payload = {
        "company": deal.company_name,
        "contract_review": contract.model_dump(),
        "due_diligence": diligence.model_dump(),
        "financial_model": financials.model_dump(),
    }
    raw = _call(BOARD_SYSTEM, json.dumps(payload), "BoardMemo", BoardMemo.model_json_schema())
    return BoardMemo.model_validate_json(raw)


def run(deal: DealInput, threshold: float = _CONFIDENCE_THRESHOLD) -> DealRoomResult:
    """Run the confidence-gated deal room pipeline.

    Halts and returns an EscalationFlag if any stage confidence falls below threshold.
    Returns a completed DealRoomResult with board_memo when all stages pass.
    """
    contract = _review_contract(deal)
    if contract.confidence < threshold:
        return DealRoomResult(
            company_name=deal.company_name,
            completed=False,
            escalation=EscalationFlag(
                stage="contract_review",
                confidence=contract.confidence,
                threshold=threshold,
                reason=(
                    f"Contract confidence {contract.confidence:.2f} below threshold {threshold}. "
                    "Insufficient contract text to proceed."
                ),
            ),
            contract_review=contract,
        )

    diligence = _run_due_diligence(deal)
    if diligence.confidence < threshold:
        return DealRoomResult(
            company_name=deal.company_name,
            completed=False,
            escalation=EscalationFlag(
                stage="due_diligence",
                confidence=diligence.confidence,
                threshold=threshold,
                reason=(
                    f"Due diligence confidence {diligence.confidence:.2f} below threshold {threshold}. "
                    "Document coverage insufficient."
                ),
            ),
            contract_review=contract,
            due_diligence=diligence,
        )

    financials = _build_financial_model(deal)
    if financials.confidence < threshold:
        return DealRoomResult(
            company_name=deal.company_name,
            completed=False,
            escalation=EscalationFlag(
                stage="financial_model",
                confidence=financials.confidence,
                threshold=threshold,
                reason=(
                    f"Financial model confidence {financials.confidence:.2f} below threshold {threshold}. "
                    "Insufficient financial data."
                ),
            ),
            contract_review=contract,
            due_diligence=diligence,
            financial_model=financials,
        )

    memo = _write_board_memo(deal, contract, diligence, financials)
    return DealRoomResult(
        company_name=deal.company_name,
        completed=True,
        contract_review=contract,
        due_diligence=diligence,
        financial_model=financials,
        board_memo=memo,
    )


print("Workflow defined.")
print(f"Default confidence threshold: {_CONFIDENCE_THRESHOLD}")
print("Gate stages: contract_review, due_diligence, financial_model")

---

## Part 5 — Sample 1: Clean Deal (NovaTech Software Inc)

NovaTech is a well-documented SaaS acquisition:

- **Contract**: Full asset purchase agreement with IP assignment, non-compete, escrow, and standard reps & warranties at 100% cap
- **Diligence**: Audited financials, SOC 2 Type II, no open litigation, stable management team
- **Financials**: ARR $6.1m growing 28% YoY, EBITDA positive, zero debt, three years of audited accounts

All three stages should pass the confidence gate. The pipeline should reach the board memo and return `completed=True`.

In [ ]:
CLEAN_DEAL = DealInput(
    company_name="NovaTech Software Inc",
    contract_text=(
        "Asset Purchase Agreement. Seller: NovaTech Software Inc. Buyer: Vertex Partners LLC. "
        "Purchase price: $18m. IP fully assigned to buyer. Non-compete 3 years, 50-mile radius. "
        "Representations and warranties standard SaaS; cap at 100% of purchase price, 24-month tail. "
        "Escrow: 10% held 12 months. Indemnification for pre-closing liabilities. "
        "MAC clause. Governing law: Delaware."
    ),
    diligence_documents=[
        "Audited FY2025 financials: ARR $6.1m, Net Revenue Retention 118%, churn 4% annual. "
        "Top customer 14% of ARR. No customer concentration risk.",
        "Management team stable 3+ years. CTO and VP Sales both have retention packages tied to deal close.",
        "No open litigation. IP fully owned; no third-party licenses embedded in product. "
        "SOC 2 Type II certified. GDPR compliant.",
    ],
    financial_summary=(
        "ARR $6.1m, growing 28% YoY. EBITDA positive at $1.2m. Zero debt. "
        "Full audited financials available for past 3 years. Strong cash conversion."
    ),
)

print(f"Running deal room pipeline for: {CLEAN_DEAL.company_name}")
print("Stages: contract_review -> due_diligence -> financial_model -> board_memo")
print()

clean_result = run(CLEAN_DEAL)

print(f"Completed: {clean_result.completed}")
print()

if clean_result.completed:
    memo = clean_result.board_memo
    print(f"RECOMMENDATION: {memo.recommended_position.upper()}")
    print(f"Verdict: {memo.one_sentence_verdict}")
    print()
    print("Key Risks:")
    for risk in memo.key_risks:
        print(f"  * {risk}")
    if memo.conditions_to_proceed:
        print()
        print("Conditions to Proceed:")
        for cond in memo.conditions_to_proceed:
            print(f"  * {cond}")
    print()
    print(f"Pipeline Confidence: {memo.confidence:.2f}")
    print()
    print("Stage Confidence Scores:")
    print(f"  Contract Review:  {clean_result.contract_review.confidence:.2f}")
    print(f"  Due Diligence:    {clean_result.due_diligence.confidence:.2f}")
    print(f"  Financial Model:  {clean_result.financial_model.confidence:.2f}")
    print()
    print("Executive Summary:")
    print(memo.executive_summary)
else:
    print(f"PIPELINE HALTED at stage: {clean_result.escalation.stage}")
    print(f"Confidence: {clean_result.escalation.confidence:.2f} (threshold: {clean_result.escalation.threshold})")
    print(f"Reason: {clean_result.escalation.reason}")

---

## Part 6 — Sample 2: Messy Deal (Acme Logistics Ltd)

Acme Logistics is a more problematic package:

- **Contract**: Standard SPA but missing an IP assignment clause and non-compete; warranty cap only 30%
- **Diligence**: Three documents — management accounts with heavy customer concentration (71% from top 3), key-person departures (CFO vacancy, operations manager leaving), and an open HMRC enquiry with no provision
- **Financials**: Management accounts only (no audit), no audited figures for prior years

The thin diligence coverage should score below the 0.6 threshold and trigger the escalation halt, demonstrating the core value of the gate: the pipeline stops before spending tokens on a financial model or board memo based on unreliable inputs.

In [ ]:
ACME_DEAL = DealInput(
    company_name="Acme Logistics Ltd",
    contract_text=(
        "Share Purchase Agreement dated 1 June 2026. Seller: Acme Holdings plc. Buyer: Meridian Capital. "
        "Purchase price: £24m. Completion accounts mechanism. Warranties: standard business warranties, "
        "capped at 30% of purchase price, 18-month tail. No IP assignment clause. No non-compete. "
        "Indemnity for known tax liability of £1.2m. Material adverse change clause included. "
        "Governing law: England & Wales."
    ),
    diligence_documents=[
        "FY2025 management accounts: Revenue £8.2m, EBITDA £1.4m, Net debt £2.1m. "
        "Three top customers represent 71% of revenue. Two customer contracts expire Dec 2026 without renewal options.",
        "CEO joined 18 months ago; CFO vacancy unfilled for 4 months. "
        "Key operations manager handed notice 2 weeks ago.",
        "HMRC enquiry open since March 2025 re: R&D tax credit claim of £340k. "
        "No provision taken in accounts. Legal counsel rates resolution probability at 60%.",
    ],
    financial_summary=(
        "Revenue £8.2m growing at ~12% p.a. EBITDA margin 17%, improving from 14% two years ago. "
        "Net debt £2.1m. Capex requirements low (asset-light model). No audited accounts -- management accounts only."
    ),
)

print(f"Running deal room pipeline for: {ACME_DEAL.company_name}")
print("Stages: contract_review -> due_diligence -> financial_model -> board_memo")
print("Expected: pipeline should halt at due_diligence due to low confidence")
print()

acme_result = run(ACME_DEAL)

print(f"Completed: {acme_result.completed}")
print()

if not acme_result.completed:
    flag = acme_result.escalation
    print(f"PIPELINE HALTED at stage: {flag.stage}")
    print(f"Confidence: {flag.confidence:.2f} (threshold: {flag.threshold})")
    print(f"Reason: {flag.reason}")
    print()
    print("Partial results available:")
    if acme_result.contract_review:
        cr = acme_result.contract_review
        print(f"  Contract Review (confidence {cr.confidence:.2f}): {len(cr.risk_findings)} risk findings")
        print(f"  Summary: {cr.summary}")
    if acme_result.due_diligence:
        dd = acme_result.due_diligence
        print(f"  Due Diligence (confidence {dd.confidence:.2f}): {len(dd.risks)} risks, {len(dd.deal_breakers)} deal-breakers")
        if dd.deal_breakers:
            print("  Deal-breakers flagged:")
            for db in dd.deal_breakers:
                print(f"    - {db}")
    print()
    print("Board memo: not generated (pipeline halted before this stage)")
else:
    print(f"RECOMMENDATION: {acme_result.board_memo.recommended_position.upper()}")
    print(f"Verdict: {acme_result.board_memo.one_sentence_verdict}")

---

## Starter Exercise

### Part A — Raise the threshold

Modify the confidence threshold to **0.75** and re-run both deals.

- Which additional stages halt?
- Does NovaTech still complete the pipeline?
- What does this tell you about calibrating the threshold for conservative vs. aggressive deal desks?

```python
# Hint: run() accepts a threshold parameter
result = run(CLEAN_DEAL, threshold=0.75)
```

### Part B — Add a fourth stage: `IntegrationPlan`

Add a new stage that generates a **100-day integration plan**, but only when the board memo recommends `'proceed'`.

The `IntegrationPlan` schema should include:
- `company_name: str`
- `day_1_actions: list[str]` — immediate actions on close (legal, comms, IT)
- `day_30_milestones: list[str]` — first month targets
- `day_100_milestones: list[str]` — 100-day targets
- `key_risks: list[str]` — integration-specific risks (people, systems, customers)
- `owner: str` — recommended integration lead role

Add an `integration_plan: IntegrationPlan | None` field to `DealRoomResult` and extend the `run()` function to call the integration planner only when `memo.recommended_position == 'proceed'`.

The system prompt for this stage should instruct the model to synthesise from the board memo and due diligence findings.

In [ ]:
# Answer Key

# Part A -- Raise the threshold to 0.75

HIGHER_THRESHOLD = 0.75

print(f"=== NovaTech at threshold {HIGHER_THRESHOLD} ===")
clean_strict = run(CLEAN_DEAL, threshold=HIGHER_THRESHOLD)
if clean_strict.completed:
    print("Completed: True")
    print(f"Recommendation: {clean_strict.board_memo.recommended_position.upper()}")
    print(f"Pipeline confidence: {clean_strict.board_memo.confidence:.2f}")
else:
    flag = clean_strict.escalation
    print(f"Completed: False -- halted at {flag.stage} (confidence {flag.confidence:.2f})")

print()
print(f"=== Acme Logistics at threshold {HIGHER_THRESHOLD} ===")
acme_strict = run(ACME_DEAL, threshold=HIGHER_THRESHOLD)
if not acme_strict.completed:
    flag = acme_strict.escalation
    print(f"Completed: False -- halted at {flag.stage} (confidence {flag.confidence:.2f})")
else:
    print("Completed: True")


# Part B -- IntegrationPlan stage

INTEGRATION_SYSTEM = (
    "You are a post-merger integration specialist. Based on the board memo and due diligence findings provided, "
    "generate a 100-day integration plan as an IntegrationPlan JSON object.\n\n"
    "Rules:\n"
    "- day_1_actions: legal entity transfer, employee comms, IT access, customer notification\n"
    "- day_30_milestones: first HR reviews, system integration kick-off, customer retention calls\n"
    "- day_100_milestones: synergy targets, product roadmap merge, financial reporting consolidated\n"
    "- key_risks: people retention, system incompatibility, customer churn, culture clash\n"
    "- owner: recommend the most appropriate integration lead role for this deal size\n"
    "- Return ONLY valid JSON matching the schema -- no prose."
)


class IntegrationPlan(BaseModel):
    company_name: str = Field(description="Target company name")
    day_1_actions: list[str] = Field(description="Immediate actions on close: legal, comms, IT")
    day_30_milestones: list[str] = Field(description="First month targets")
    day_100_milestones: list[str] = Field(description="100-day targets")
    key_risks: list[str] = Field(description="Integration-specific risks: people, systems, customers")
    owner: str = Field(description="Recommended integration lead role")


class DealRoomResultV2(BaseModel):
    company_name: str
    completed: bool
    escalation: EscalationFlag | None = None
    contract_review: ContractReview | None = None
    due_diligence: DueDiligenceReport | None = None
    financial_model: FinancialModel | None = None
    board_memo: BoardMemo | None = None
    integration_plan: IntegrationPlan | None = None


def _build_integration_plan(
    deal: DealInput,
    memo: BoardMemo,
    diligence: DueDiligenceReport,
) -> IntegrationPlan:
    payload = {
        "company": deal.company_name,
        "board_memo": memo.model_dump(),
        "due_diligence_summary": diligence.summary,
        "deal_breakers_cleared": diligence.deal_breakers,
    }
    raw = _call(
        INTEGRATION_SYSTEM,
        json.dumps(payload),
        "IntegrationPlan",
        IntegrationPlan.model_json_schema(),
    )
    return IntegrationPlan.model_validate_json(raw)


def run_v2(deal: DealInput, threshold: float = _CONFIDENCE_THRESHOLD) -> DealRoomResultV2:
    """Extended pipeline: adds integration_plan stage when board recommends 'proceed'."""
    contract = _review_contract(deal)
    if contract.confidence < threshold:
        return DealRoomResultV2(
            company_name=deal.company_name,
            completed=False,
            escalation=EscalationFlag(
                stage="contract_review",
                confidence=contract.confidence,
                threshold=threshold,
                reason=f"Contract confidence {contract.confidence:.2f} below threshold {threshold}.",
            ),
            contract_review=contract,
        )

    diligence = _run_due_diligence(deal)
    if diligence.confidence < threshold:
        return DealRoomResultV2(
            company_name=deal.company_name,
            completed=False,
            escalation=EscalationFlag(
                stage="due_diligence",
                confidence=diligence.confidence,
                threshold=threshold,
                reason=f"Due diligence confidence {diligence.confidence:.2f} below threshold {threshold}.",
            ),
            contract_review=contract,
            due_diligence=diligence,
        )

    financials = _build_financial_model(deal)
    if financials.confidence < threshold:
        return DealRoomResultV2(
            company_name=deal.company_name,
            completed=False,
            escalation=EscalationFlag(
                stage="financial_model",
                confidence=financials.confidence,
                threshold=threshold,
                reason=f"Financial model confidence {financials.confidence:.2f} below threshold {threshold}.",
            ),
            contract_review=contract,
            due_diligence=diligence,
            financial_model=financials,
        )

    memo = _write_board_memo(deal, contract, diligence, financials)

    # Fourth stage: integration plan only when board recommends proceed
    integration = None
    if memo.recommended_position == "proceed":
        integration = _build_integration_plan(deal, memo, diligence)

    return DealRoomResultV2(
        company_name=deal.company_name,
        completed=True,
        contract_review=contract,
        due_diligence=diligence,
        financial_model=financials,
        board_memo=memo,
        integration_plan=integration,
    )


print()
print("=== Part B: run_v2 on NovaTech (integration plan if proceed) ===")
clean_v2 = run_v2(CLEAN_DEAL)
print(f"Completed: {clean_v2.completed}")
if clean_v2.board_memo:
    print(f"Board recommendation: {clean_v2.board_memo.recommended_position.upper()}")
if clean_v2.integration_plan:
    plan = clean_v2.integration_plan
    print(f"Integration plan generated. Owner: {plan.owner}")
    print("Day 1 actions:")
    for action in plan.day_1_actions:
        print(f"  - {action}")
    print("Day 100 milestones:")
    for milestone in plan.day_100_milestones:
        print(f"  - {milestone}")
else:
    print("Integration plan: not generated (board did not recommend proceed)")